# Imports

In [ ]:
import kagglehub
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import gc
import pandas as pd

import torch
import torch.nn as nn

from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import timm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

seed=8359
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Globals

In [ ]:
KAGGLE_DATASET_NAME="ciccionoss/rrdataset-split-55-10-35"

MODEL_NAME="vit_base_patch16_224"
HIDDEN_DIM = 256
DROPOUT = 0.3
ACTIVATION_NAME = "silu"

HR_LEARNING_RATES = 3e-5
HR_WEIGHT_DECAYS = 1e-3
HR_LAMBDA_TRANSFORMS = [0.1, 0.25, 0.5, 0.75, 0.9]
HR_BATCH_SIZES = 32
HR_NUM_EPOCHS = 10
HR_PATIENCE = 5

LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-3
LAMBDA_TRANSFORM = 0.25
BATCH_SIZE = 32
NUM_EPOCHS = 20
PATIENCE = 7

NUM_WORKERS = 0
NUM_WORKERS_TEST = 2

MODEL_SAVE_PATH=os.path.join(os.getcwd(), "models")
CSV_SAVE_PATH=os.path.join(os.getcwd(), "training_history")

In [ ]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Device: {device}")

# Utils

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.root = root
        self.samples = []

        real_ai_map = {
            "real": 0,
            "ai": 1
        }

        transformation_map = {
            "original": 0,
            "redigital": 1,
            "transfer": 2
        }

        for transform_type in os.listdir(root):
            transform_path = os.path.join(root, transform_type)

            if not os.path.isdir(transform_path):
                continue

            for class_type in os.listdir(transform_path):
                class_path = os.path.join(transform_path, class_type)

                if not os.path.isdir(class_path):
                    continue

                for image_name in os.listdir(class_path):
                    image_path = os.path.join(class_path, image_name)

                    if transform_type not in transformation_map:
                        continue

                    if class_type not in real_ai_map:
                        continue

                    transformation_label = transformation_map[transform_type]
                    real_ai_label = real_ai_map[class_type]

                    self.samples.append(
                        (image_path, real_ai_label, transformation_label)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, real_ai_label, transformation_label = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, real_ai_label, transformation_label

    def __str__(self):
        return f"This dataset has {len(self.samples)} samples"

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
def create_dataloaders(train_data, val_data, test_data, batch_size, dvc, num_work):
    pin_memory = True if dvc == "cuda" else False

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        dataset=val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, test_loader

# Data

In [ ]:
print(f"Downloading {KAGGLE_DATASET_NAME} dataset!")
path = kagglehub.dataset_download(KAGGLE_DATASET_NAME)
print(f"Dataset downloaded at {path}!")

In [ ]:
path = os.path.join(os.getcwd(), r"\rrdataset-split-55-10-35\versions\1")

In [ ]:
path = os.path.join(os.getcwd(), "archive")

In [ ]:
dataset_train = CustomDataset(
    root=path + "/train",
    transform=data_transform
)

dataset_val = CustomDataset(
    root=path + "/val",
    transform=data_transform
)

dataset_test = CustomDataset(
    root=path + "/test",
    transform=data_transform
)
print(f"Dataset train: {dataset_train}, Dataset val: {dataset_val}, Dataset test: {dataset_test}")

In [ ]:
real_ai_names = {
    0: "real",
    1: "ai"
}

transformation_names = {
    0: "original",
    1: "redigital",
    2: "transfer"
}

def dataset_balance_table(dataset):
    samples_df = pd.DataFrame(
        dataset.samples,
        columns=["image_path", "real_ai_label", "transformation_label"]
    )
    samples_df["real_ai"] = samples_df["real_ai_label"].map(real_ai_names)
    samples_df["transformation"] = samples_df["transformation_label"].map(transformation_names)

    balance_table = pd.crosstab(
        samples_df["transformation"],
        samples_df["real_ai"]
    ).reindex(
        index=list(transformation_names.values()),
        columns=list(real_ai_names.values()),
        fill_value=0
    )
    balance_table["total"] = balance_table.sum(axis=1)

    total_row = balance_table.sum(axis=0).to_frame().T
    total_row.index = ["total"]

    return pd.concat([balance_table, total_row])

datasets = {
    "train": dataset_train,
    "val": dataset_val,
    "test": dataset_test
}

balance_summary = []
balance_tables = {}

for split_name, dataset in datasets.items():
    balance_table = dataset_balance_table(dataset)
    joint_counts = balance_table.loc[
        list(transformation_names.values()),
        list(real_ai_names.values())
    ]
    min_joint_count = int(joint_counts.to_numpy().min())
    max_joint_count = int(joint_counts.to_numpy().max())
    max_joint_gap = max_joint_count - min_joint_count

    balance_summary.append({
        "split": split_name,
        "samples": len(dataset),
        "min_joint_count": min_joint_count,
        "max_joint_count": max_joint_count,
        "max_joint_gap": max_joint_gap,
        "balanced": max_joint_gap <= 1
    })
    balance_tables[split_name] = balance_table

balance_summary = pd.DataFrame(balance_summary)
display(balance_summary)

for split_name, balance_table in balance_tables.items():
    print(f"{split_name} joint distribution")
    display(balance_table)

In [ ]:
print("Train balance")
display(dataset_balance_table(dataset_train))

print("Validation balance")
display(dataset_balance_table(dataset_val))

print("Test balance")
display(dataset_balance_table(dataset_test))

# Network

In [ ]:
class MultiHeadModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        num_transform_classes=3,
        hidden_dim=256,
        dropout=0.3,
        activation_name="gelu"
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        match activation_name:
            case "gelu":
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()
            case "relu":
                activation_real_ai = nn.ReLU()
                activation_transform = nn.ReLU()
            case "leaky_relu":
                activation_real_ai = nn.LeakyReLU(0.01)
                activation_transform = nn.LeakyReLU(0.01)
            case "silu":
                activation_real_ai = nn.SiLU()
                activation_transform = nn.SiLU()
            case _:
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()

        self.real_ai_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_real_ai,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

        self.transformation_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_transform,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_transform_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        real_ai_logits = self.real_ai_head(features).squeeze(1)
        transformation_logits = self.transformation_head(features)

        return real_ai_logits, transformation_logits

In [ ]:
class SingleHeadBinaryModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        self.real_ai_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        features = self.backbone(x)

        real_ai_logits = self.real_ai_head(features).squeeze(1)

        return real_ai_logits

In [ ]:
class SingleHeadTransformartionModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        num_transform_classes=3,
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        self.transformation_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_transform_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        transformation_logits = self.transformation_head(features)

        return transformation_logits

# Train

In [ ]:
def evaluate_model(model, data_loader, lambda_transform, device):
    model.eval()

    bce_loss = nn.BCEWithLogitsLoss()
    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0
    total_samples = 0

    all_fake_labels = []
    all_fake_preds = []

    all_transform_labels = []
    all_transform_preds = []

    with torch.no_grad():
        for images, fake_labels, transform_labels in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            fake_labels = fake_labels.to(device)
            transform_labels = transform_labels.to(device)

            fake_logits, transform_logits = model(images)

            loss_fake = bce_loss(fake_logits, fake_labels.float())
            loss_transform = ce_loss(transform_logits, transform_labels)

            loss = (1 - lambda_transform) * loss_fake + lambda_transform * loss_transform

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
            transform_preds = torch.argmax(transform_logits, dim=1)

            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_preds.extend(fake_preds.cpu().numpy())

            all_transform_labels.extend(transform_labels.cpu().numpy())
            all_transform_preds.extend(transform_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    real_ai_acc = accuracy_score(
        all_fake_labels, 
        all_fake_preds
    )
    real_ai_f1 = f1_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )

    transform_acc = accuracy_score(
        all_transform_labels,
        all_transform_preds
    )
    transform_macro_f1 = f1_score(
        all_transform_labels,
        all_transform_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "real_ai_acc": real_ai_acc,
        "real_ai_f1": real_ai_f1,
        "transform_acc": transform_acc,
        "transform_macro_f1": transform_macro_f1
    }

In [ ]:
def train_model(
    model_name,
    learning_rate,
    weight_decay,
    lambda_transform,
    batch_size,
    hidden_dim,
    dropout,
    activation_name,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = MultiHeadModel(
      model_name=model_name,
      num_transform_classes=3,
      hidden_dim=hidden_dim,
      dropout=dropout,
      activation_name=activation_name
  ).to(device)

  bce_loss = nn.BCEWithLogitsLoss()
  ce_loss = nn.CrossEntropyLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"lt-{lambda_transform}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"a-{activation_name}_"
    + f"seed-{seed}_MultiHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_fake_labels = []
    all_train_fake_preds = []

    all_train_transform_labels = []
    all_train_transform_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, fake_labels, transform_labels in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"LT={lambda_transform} | BS={batch_size} | "
            f"Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      fake_labels = fake_labels.to(device)
      transform_labels = transform_labels.to(device)

      fake_logits, transform_logits = model(images)

      loss_fake = bce_loss(fake_logits, fake_labels.float())
      loss_transform = ce_loss(transform_logits, transform_labels)

      loss = (1 - lambda_transform) * loss_fake + lambda_transform * loss_transform

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss.item() * batch_size_current
      train_total_samples += batch_size_current

      fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
      transform_preds = torch.argmax(transform_logits, dim=1)

      all_train_fake_labels.extend(fake_labels.cpu().numpy())
      all_train_fake_preds.extend(fake_preds.cpu().numpy())

      all_train_transform_labels.extend(transform_labels.cpu().numpy())
      all_train_transform_preds.extend(transform_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_real_ai_acc = accuracy_score(
        all_train_fake_labels,
        all_train_fake_preds
    )
    train_real_ai_f1 = f1_score(
        all_train_fake_labels,
        all_train_fake_preds,
        zero_division=0
    )
    train_transform_acc = accuracy_score(
        all_train_transform_labels,
        all_train_transform_preds
    )
    train_transform_macro_f1 = f1_score(
        all_train_transform_labels,
        all_train_transform_preds,
        average="macro",
        zero_division=0
    )

    val_metrics = evaluate_model(
        model,
        val_loader,
        lambda_transform,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "lambda_transform": lambda_transform,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "activation": activation_name,

        "train_loss": avg_train_loss,
        "train_real_ai_acc": train_real_ai_acc,
        "train_real_ai_f1": train_real_ai_f1,
        "train_transform_acc": train_transform_acc,
        "train_transform_macro_f1": train_transform_macro_f1,

        "val_loss": val_metrics["loss"],
        "val_real_ai_acc": val_metrics["real_ai_acc"],
        "val_real_ai_f1": val_metrics["real_ai_f1"],
        "val_transform_acc": val_metrics["transform_acc"],
        "val_transform_macro_f1": val_metrics["transform_macro_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Real AI Acc: {train_real_ai_acc:.4f} | Train Real AI F1: {train_real_ai_f1:.4f} | "
        f"Train Transform Acc: {train_transform_acc:.4f} | Train Transform F1: {train_transform_macro_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Real AI Acc: {val_metrics['real_ai_acc']:.4f} | Val Real AI F1: {val_metrics['real_ai_f1']:.4f} | "
        f"Val Transform Acc: {val_metrics['transform_acc']:.4f} | Val Transform F1: {val_metrics['transform_macro_f1']:.4f}"
    )

    current_score = (
      0.5 * val_metrics["real_ai_acc"]
      + 0.5 * val_metrics["transform_acc"]
    )

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

In [ ]:
def evaluate_model_binary(model, data_loader, device):
    model.eval()

    bce_loss = nn.BCEWithLogitsLoss()

    total_loss = 0
    total_samples = 0

    all_fake_labels = []
    all_fake_preds = []

    with torch.no_grad():
        for images, fake_labels, _ in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            fake_labels = fake_labels.to(device)

            fake_logits = model(images)

            loss_fake = bce_loss(fake_logits, fake_labels.float())

            batch_size = images.size(0)

            total_loss += loss_fake.item() * batch_size
            total_samples += batch_size

            fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()

            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_preds.extend(fake_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    real_ai_acc = accuracy_score(all_fake_labels, all_fake_preds)
    real_ai_f1 = f1_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "real_ai_acc": real_ai_acc,
        "real_ai_f1": real_ai_f1
    }

In [ ]:
def train_model_binary(
    model_name,
    learning_rate,
    weight_decay,
    batch_size,
    hidden_dim,
    dropout,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = SingleHeadBinaryModel(
      model_name=model_name,
      hidden_dim=hidden_dim,
      dropout=dropout
  ).to(device)

  bce_loss = nn.BCEWithLogitsLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"seed-{seed}_BinaryHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_fake_labels = []
    all_train_fake_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, fake_labels, _ in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"BS={batch_size} | Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      fake_labels = fake_labels.to(device)

      fake_logits = model(images)

      loss_fake = bce_loss(fake_logits, fake_labels.float())

      optimizer.zero_grad()
      loss_fake.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss_fake.item() * batch_size_current
      train_total_samples += batch_size_current

      fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()

      all_train_fake_labels.extend(fake_labels.cpu().numpy())
      all_train_fake_preds.extend(fake_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_real_ai_acc = accuracy_score(
        all_train_fake_labels,
        all_train_fake_preds
    )
    train_real_ai_f1 = f1_score(
        all_train_fake_labels,
        all_train_fake_preds,
        zero_division=0
    )

    val_metrics = evaluate_model_binary(
        model,
        val_loader,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,

        "train_loss": avg_train_loss,
        "train_real_ai_acc": train_real_ai_acc,
        "train_real_ai_f1": train_real_ai_f1,

        "val_loss": val_metrics["loss"],
        "val_real_ai_acc": val_metrics["real_ai_acc"],
        "val_real_ai_f1": val_metrics["real_ai_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_real_ai_acc:.4f} | Train F1: {train_real_ai_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['real_ai_acc']:.4f} | Val F1: {val_metrics['real_ai_f1']:.4f}"
    )

    current_score = val_metrics["real_ai_acc"]

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

In [ ]:
def evaluate_model_transformation(model, data_loader, device):
    model.eval()

    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0
    total_samples = 0

    all_transform_labels = []
    all_transform_preds = []

    with torch.no_grad():
        for images, _, transform_labels in tqdm(
            data_loader,
            desc="Validation",
            disable=True
          ):
            images = images.to(device)
            transform_labels = transform_labels.to(device)

            transform_logits = model(images)

            loss_transform = ce_loss(transform_logits, transform_labels)

            batch_size = images.size(0)

            total_loss += loss_transform.item() * batch_size
            total_samples += batch_size

            transform_preds = torch.argmax(transform_logits, dim=1)

            all_transform_labels.extend(transform_labels.cpu().numpy())
            all_transform_preds.extend(transform_preds.cpu().numpy())

    avg_loss = total_loss / total_samples

    transform_acc = accuracy_score(
        all_transform_labels,
        all_transform_preds
    )
    transform_macro_f1 = f1_score(
        all_transform_labels,
        all_transform_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "transform_acc": transform_acc,
        "transform_macro_f1": transform_macro_f1
    }

In [ ]:
def train_model_transformation(
    model_name,
    learning_rate,
    weight_decay,
    batch_size,
    hidden_dim,
    dropout,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = SingleHeadTransformartionModel(
      model_name=model_name,
      num_transform_classes=3,
      hidden_dim=hidden_dim,
      dropout=dropout
  ).to(device)

  ce_loss = nn.CrossEntropyLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"seed-{seed}_TransformationHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_total_samples = 0

    all_train_transform_labels = []
    all_train_transform_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, _, transform_labels in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"BS={batch_size} | Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      transform_labels = transform_labels.to(device)

      transform_logits = model(images)

      loss_transform = ce_loss(transform_logits, transform_labels)

      optimizer.zero_grad()
      loss_transform.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss_transform.item() * batch_size_current
      train_total_samples += batch_size_current

      transform_preds = torch.argmax(transform_logits, dim=1)

      all_train_transform_labels.extend(transform_labels.cpu().numpy())
      all_train_transform_preds.extend(transform_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples

    train_transform_acc = accuracy_score(
        all_train_transform_labels,
        all_train_transform_preds
    )
    train_transform_macro_f1 = f1_score(
        all_train_transform_labels,
        all_train_transform_preds,
        average="macro",
        zero_division=0
    )

    val_metrics = evaluate_model_transformation(
        model,
        val_loader,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,

        "train_loss": avg_train_loss,
        "train_transform_acc": train_transform_acc,
        "train_transform_macro_f1": train_transform_macro_f1,

        "val_loss": val_metrics["loss"],
        "val_transform_acc": val_metrics["transform_acc"],
        "val_transform_macro_f1": val_metrics["transform_macro_f1"]
    }

    history.append(epoch_result)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_transform_acc:.4f} | Train F1: {train_transform_macro_f1:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['transform_acc']:.4f} | Val F1: {val_metrics['transform_macro_f1']:.4f}"
    )

    current_score = val_metrics["transform_acc"]

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

With Hyper-Parameter Research

In [ ]:
for lt in HR_LAMBDA_TRANSFORMS:
    print(f"\n\nStarting training with lambda_transform={lt}...\n")
    result = train_model(
        model_name=MODEL_NAME,
        learning_rate=HR_LEARNING_RATES,
        weight_decay=HR_WEIGHT_DECAYS,
        lambda_transform=lt,
        batch_size=HR_BATCH_SIZES,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        activation_name=ACTIVATION_NAME,
        num_epochs=HR_NUM_EPOCHS,
        patience=HR_PATIENCE
    )
    print(
        f"Best Epoch: {result['best_epoch']} | "
        f"Best Score: {result['best_score']:.4f} | "
        f"Best Val Metrics: {result['best_val_metrics']}"
    )

Without Hyper-Parameter Research

In [ ]:
best_result_no_hr = train_model(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lambda_transform=LAMBDA_TRANSFORM,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    activation_name=ACTIVATION_NAME,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)

best_result_binary_no_hr = train_model_binary(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)

best_result_transformation_no_hr = train_model_transformation(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)

In [ ]:
history_multihead = best_result_no_hr["history"]
history_binary = best_result_binary_no_hr["history"]
history_transformation = best_result_transformation_no_hr["history"]

df_history_multihead = pd.DataFrame(history_multihead)
df_history_binary = pd.DataFrame(history_binary)
df_history_transformation = pd.DataFrame(history_transformation)

print("MultiHead Model Training History:")
print(df_history_multihead.head())
print(f"\nShape: {df_history_multihead.shape}")

print("\n\nBinary Model Training History:")
print(df_history_binary.head())
print(f"\nShape: {df_history_binary.shape}")

print("\n\nTransformation Model Training History:")
print(df_history_transformation.head())
print(f"\nShape: {df_history_transformation.shape}")

csv_save_path = CSV_SAVE_PATH
if not os.path.exists(csv_save_path):
    os.makedirs(csv_save_path)

df_history_multihead.to_csv(os.path.join(csv_save_path, "history_multihead.csv"), index=False)
df_history_binary.to_csv(os.path.join(csv_save_path, "history_binary.csv"), index=False)
df_history_transformation.to_csv(os.path.join(csv_save_path, "history_transformation.csv"), index=False)

print(f"\n\nTraining histories saved to {csv_save_path}")

# Testing

Create the data loader for test set

In [ ]:
_, _, test_loader = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    BATCH_SIZE,
    device,
    NUM_WORKERS
  )

In [ ]:
# Helper functions for test evaluation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

REAL_AI_NAMES = {
    0: "real",
    1: "ai"
}

TRANSFORMATION_NAMES = {
    0: "original",
    1: "redigital",
    2: "transfer"
}


def safe_accuracy(y_true, y_pred):
    if len(y_true) == 0:
        return None

    return float(accuracy_score(y_true, y_pred))


def calculate_real_ai_accuracy_splits(
    real_ai_labels,
    real_ai_preds,
    transformation_labels
):
    real_ai_labels = np.asarray(real_ai_labels).astype(int).reshape(-1)
    real_ai_preds = np.asarray(real_ai_preds).astype(int).reshape(-1)
    transformation_labels = np.asarray(transformation_labels).astype(int).reshape(-1)

    accuracy_by_transformation = {}

    for transform_id, transform_name in TRANSFORMATION_NAMES.items():
        mask = transformation_labels == transform_id

        accuracy_by_transformation[transform_name] = {
            "samples": int(mask.sum()),
            "accuracy": safe_accuracy(
                real_ai_labels[mask],
                real_ai_preds[mask]
            )
        }

    accuracy_by_class_and_transformation = {}

    for class_id, class_name in REAL_AI_NAMES.items():
        accuracy_by_class_and_transformation[class_name] = {}

        for transform_id, transform_name in TRANSFORMATION_NAMES.items():
            mask = (
                (real_ai_labels == class_id)
                & (transformation_labels == transform_id)
            )

            accuracy_by_class_and_transformation[class_name][transform_name] = {
                "samples": int(mask.sum()),
                "accuracy": safe_accuracy(
                    real_ai_labels[mask],
                    real_ai_preds[mask]
                )
            }

    return accuracy_by_transformation, accuracy_by_class_and_transformation


def calculate_transformation_accuracy_splits(
    transformation_labels,
    transformation_preds
):
    transformation_labels = np.asarray(transformation_labels).astype(int).reshape(-1)
    transformation_preds = np.asarray(transformation_preds).astype(int).reshape(-1)

    accuracy_by_true_transformation = {}

    for transform_id, transform_name in TRANSFORMATION_NAMES.items():
        mask = transformation_labels == transform_id

        accuracy_by_true_transformation[transform_name] = {
            "samples": int(mask.sum()),
            "accuracy": safe_accuracy(
                transformation_labels[mask],
                transformation_preds[mask]
            )
        }

    return accuracy_by_true_transformation


def print_real_ai_split_results(
    title,
    accuracy_by_transformation,
    accuracy_by_class_and_transformation
):
    print(f"\n{title} - Real/AI Accuracy by Transformation")
    print("-" * 80)
    print(f"{'Transformation':<20} {'Samples':<10} {'Accuracy':<10}")

    for transform_name, values in accuracy_by_transformation.items():
        acc = values["accuracy"]
        acc_text = "N/A" if acc is None else f"{acc * 100:.2f}%"

        print(
            f"{transform_name:<20} "
            f"{values['samples']:<10} "
            f"{acc_text:<10}"
        )

    print(f"\n{title} - Real/AI Accuracy by Class and Transformation")
    print("-" * 80)

    for class_name, transform_results in accuracy_by_class_and_transformation.items():
        print(f"\n{class_name.upper()} IMAGES")
        print(f"{'Transformation':<20} {'Samples':<10} {'Accuracy':<10}")

        for transform_name, values in transform_results.items():
            acc = values["accuracy"]
            acc_text = "N/A" if acc is None else f"{acc * 100:.2f}%"

            print(
                f"{transform_name:<20} "
                f"{values['samples']:<10} "
                f"{acc_text:<10}"
            )


def print_transformation_split_results(
    title,
    accuracy_by_true_transformation
):
    print(f"\n{title} - Transformation Accuracy by True Transformation")
    print("-" * 80)
    print(f"{'True transformation':<20} {'Samples':<10} {'Accuracy':<10}")

    for transform_name, values in accuracy_by_true_transformation.items():
        acc = values["accuracy"]
        acc_text = "N/A" if acc is None else f"{acc * 100:.2f}%"

        print(
            f"{transform_name:<20} "
            f"{values['samples']:<10} "
            f"{acc_text:<10}"
        )

Evaluate signle head binary model on the test set for deep fake detection

In [ ]:
binary_model_path = os.path.join(MODEL_SAVE_PATH, "vit_base_patch16_224_lr-3e-05_wd-0.001_bs-32_hd-256_d-0.3_seed-8359_BinaryHead.pth")

In [ ]:
loaded_model = torch.load(
    binary_model_path,
    map_location=torch.device(device)
)

model = SingleHeadBinaryModel(
    model_name=MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
)

model.load_state_dict(loaded_model)
model.to(device)

print("Binary model loaded!")

model.eval()

all_real_ai_labels = []
all_real_ai_preds = []
all_binary_transformation_labels = []

print("Starting binary model testing...")

with torch.no_grad():
    for images, real_ai_labels, transformation_labels in tqdm(
        test_loader,
        desc="Testing Binary Model",
        total=len(test_loader)
    ):
        images = images.to(device)
        real_ai_labels = real_ai_labels.to(device).float()

        real_ai_logits = model(images)
        real_ai_logits = real_ai_logits.view(-1)

        real_ai_preds = (
            torch.sigmoid(real_ai_logits) > 0.5
        ).long()

        all_real_ai_labels.extend(
            real_ai_labels.cpu().numpy().astype(int)
        )

        all_real_ai_preds.extend(
            real_ai_preds.cpu().numpy().astype(int)
        )

        all_binary_transformation_labels.extend(
            transformation_labels.cpu().numpy().astype(int)
        )


binary_real_ai_accuracy = accuracy_score(
    all_real_ai_labels,
    all_real_ai_preds
)

binary_real_ai_precision = precision_score(
    all_real_ai_labels,
    all_real_ai_preds,
    zero_division=0
)

binary_real_ai_recall = recall_score(
    all_real_ai_labels,
    all_real_ai_preds,
    zero_division=0
)

binary_real_ai_f1 = f1_score(
    all_real_ai_labels,
    all_real_ai_preds,
    zero_division=0
)

binary_real_ai_cm = confusion_matrix(
    all_real_ai_labels,
    all_real_ai_preds,
    labels=[0, 1]
)

binary_real_ai_accuracy_by_transformation, binary_real_ai_accuracy_by_class_and_transformation = (
    calculate_real_ai_accuracy_splits(
        all_real_ai_labels,
        all_real_ai_preds,
        all_binary_transformation_labels
    )
)

binary_test_metrics = {
    "model_name": "Binary",
    "real_ai_acc": binary_real_ai_accuracy,
    "real_ai_precision": binary_real_ai_precision,
    "real_ai_recall": binary_real_ai_recall,
    "real_ai_f1": binary_real_ai_f1,
    "real_ai_confusion_matrix": binary_real_ai_cm.tolist(),
    "real_ai_accuracy_by_transformation": binary_real_ai_accuracy_by_transformation,
    "real_ai_accuracy_by_class_and_transformation": binary_real_ai_accuracy_by_class_and_transformation
}

print("Testing Binary model complete!")
print(f"Real/AI Accuracy: {binary_real_ai_accuracy:.4f}")
print(f"Real/AI Precision: {binary_real_ai_precision:.4f}")
print(f"Real/AI Recall: {binary_real_ai_recall:.4f}")
print(f"Real/AI F1 Score: {binary_real_ai_f1:.4f}")

print("\nReal/AI Confusion Matrix:")
print(binary_real_ai_cm)

print_real_ai_split_results(
    "Binary Model Test",
    binary_real_ai_accuracy_by_transformation,
    binary_real_ai_accuracy_by_class_and_transformation
)

del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Evaluate signle head 3 class model on the test set for image transformation detection

In [ ]:
transformation_model_path = os.path.join(MODEL_SAVE_PATH, "vit_base_patch16_224_lr-3e-05_wd-0.001_bs-32_hd-256_d-0.3_seed-8359_TransformationHead.pth")

In [ ]:
loaded_model = torch.load(
    transformation_model_path,
    map_location=torch.device(device)
)

model = SingleHeadTransformartionModel(
    model_name=MODEL_NAME,
    num_transform_classes=3,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
)

model.load_state_dict(loaded_model)
model.to(device)

print("Transformation model loaded!")

model.eval()

all_transformation_labels = []
all_transformation_preds = []

print("Starting transformation model testing...")

with torch.no_grad():
    for images, real_ai_labels, transformation_labels in tqdm(
        test_loader,
        desc="Testing Transformation Model",
        total=len(test_loader)
    ):
        images = images.to(device)
        transformation_labels = transformation_labels.to(device)

        transformation_logits = model(images)

        transformation_preds = torch.argmax(
            transformation_logits,
            dim=1
        )

        all_transformation_labels.extend(
            transformation_labels.cpu().numpy().astype(int)
        )

        all_transformation_preds.extend(
            transformation_preds.cpu().numpy().astype(int)
        )


transformation_accuracy = accuracy_score(
    all_transformation_labels,
    all_transformation_preds
)

transformation_macro_f1 = f1_score(
    all_transformation_labels,
    all_transformation_preds,
    average="macro",
    zero_division=0
)

transformation_cm = confusion_matrix(
    all_transformation_labels,
    all_transformation_preds,
    labels=[0, 1, 2]
)

transformation_accuracy_by_true_transformation = (
    calculate_transformation_accuracy_splits(
        all_transformation_labels,
        all_transformation_preds
    )
)

transformation_test_metrics = {
    "model_name": "Transformation",
    "transform_acc": transformation_accuracy,
    "transform_macro_f1": transformation_macro_f1,
    "transform_confusion_matrix": transformation_cm.tolist(),
    "transformation_accuracy_by_true_transformation": transformation_accuracy_by_true_transformation
}

print("Testing Transformation model complete!")
print(f"Transformation Accuracy: {transformation_accuracy:.4f}")
print(f"Transformation Macro F1: {transformation_macro_f1:.4f}")

print("\nTransformation Confusion Matrix:")
print(transformation_cm)

print_transformation_split_results(
    "Single-Head Transformation Model Test",
    transformation_accuracy_by_true_transformation
)

del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Evaluate all the multi head models trained with different lamda transformation values on the test set

In [ ]:
def evaluate_multihead_models(model_path, device, lambda_value):
    loaded_model = torch.load(
        model_path,
        map_location=torch.device(device)
    )

    model = MultiHeadModel(
        model_name=MODEL_NAME,
        num_transform_classes=3,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        activation_name=ACTIVATION_NAME
    )

    model.load_state_dict(loaded_model)
    model.to(device)

    print(f"Multi-Head model with lambda={lambda_value} loaded!")

    model.eval()

    all_real_ai_labels = []
    all_real_ai_preds = []

    all_transformation_labels = []
    all_transformation_preds = []

    print("Starting Multi-Head model testing...")

    with torch.no_grad():
        for images, real_ai_labels, transformation_labels in tqdm(
            test_loader,
            desc=f"Testing Multi-Head lambda={lambda_value}",
            total=len(test_loader)
        ):
            images = images.to(device)

            real_ai_labels = real_ai_labels.to(device).float()
            transformation_labels = transformation_labels.to(device)

            real_ai_logits, transformation_logits = model(images)

            real_ai_logits = real_ai_logits.view(-1)

            real_ai_preds = (
                torch.sigmoid(real_ai_logits) > 0.5
            ).long()

            transformation_preds = torch.argmax(
                transformation_logits,
                dim=1
            )

            all_real_ai_labels.extend(
                real_ai_labels.cpu().numpy().astype(int)
            )

            all_real_ai_preds.extend(
                real_ai_preds.cpu().numpy().astype(int)
            )

            all_transformation_labels.extend(
                transformation_labels.cpu().numpy().astype(int)
            )

            all_transformation_preds.extend(
                transformation_preds.cpu().numpy().astype(int)
            )


    real_ai_accuracy = accuracy_score(
        all_real_ai_labels,
        all_real_ai_preds
    )

    real_ai_precision = precision_score(
        all_real_ai_labels,
        all_real_ai_preds,
        zero_division=0
    )

    real_ai_recall = recall_score(
        all_real_ai_labels,
        all_real_ai_preds,
        zero_division=0
    )

    real_ai_f1 = f1_score(
        all_real_ai_labels,
        all_real_ai_preds,
        zero_division=0
    )

    real_ai_cm = confusion_matrix(
        all_real_ai_labels,
        all_real_ai_preds,
        labels=[0, 1]
    )

    transformation_accuracy = accuracy_score(
        all_transformation_labels,
        all_transformation_preds
    )

    transformation_macro_f1 = f1_score(
        all_transformation_labels,
        all_transformation_preds,
        average="macro",
        zero_division=0
    )

    transformation_cm = confusion_matrix(
        all_transformation_labels,
        all_transformation_preds,
        labels=[0, 1, 2]
    )

    real_ai_accuracy_by_transformation, real_ai_accuracy_by_class_and_transformation = (
        calculate_real_ai_accuracy_splits(
            all_real_ai_labels,
            all_real_ai_preds,
            all_transformation_labels
        )
    )

    transformation_accuracy_by_true_transformation = (
        calculate_transformation_accuracy_splits(
            all_transformation_labels,
            all_transformation_preds
        )
    )

    combined_score = (
        0.8 * real_ai_accuracy
        + 0.2 * transformation_accuracy
    )

    metrics = {
        "model_name": "Multi-Head",
        "lambda_value": lambda_value,

        "real_ai_acc": real_ai_accuracy,
        "real_ai_precision": real_ai_precision,
        "real_ai_recall": real_ai_recall,
        "real_ai_f1": real_ai_f1,
        "real_ai_confusion_matrix": real_ai_cm.tolist(),

        "transform_acc": transformation_accuracy,
        "transform_macro_f1": transformation_macro_f1,
        "transform_confusion_matrix": transformation_cm.tolist(),

        "combined_score": combined_score,

        "real_ai_accuracy_by_transformation": real_ai_accuracy_by_transformation,
        "real_ai_accuracy_by_class_and_transformation": real_ai_accuracy_by_class_and_transformation,
        "transformation_accuracy_by_true_transformation": transformation_accuracy_by_true_transformation
    }

    print("Testing Multi-Head model complete!")
    print(f"Lambda value: {lambda_value}")
    print(f"Real/AI Head Accuracy: {real_ai_accuracy:.4f}")
    print(f"Real/AI Head Precision: {real_ai_precision:.4f}")
    print(f"Real/AI Head Recall: {real_ai_recall:.4f}")
    print(f"Real/AI Head F1: {real_ai_f1:.4f}")
    print(f"Transformation Head Accuracy: {transformation_accuracy:.4f}")
    print(f"Transformation Head Macro F1: {transformation_macro_f1:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    print("\nReal/AI Head Confusion Matrix:")
    print(real_ai_cm)

    print("\nTransformation Head Confusion Matrix:")
    print(transformation_cm)

    print_real_ai_split_results(
        f"Multi-Head Lambda={lambda_value} Test",
        real_ai_accuracy_by_transformation,
        real_ai_accuracy_by_class_and_transformation
    )

    print_transformation_split_results(
        f"Multi-Head Lambda={lambda_value} Test",
        transformation_accuracy_by_true_transformation
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

In [ ]:
multihead_lambda_test_results = []

for lt in HR_LAMBDA_TRANSFORMS:
    print("-" * 100)
    print(f"Evaluating Multi-Head model with lambda_transform={lt} on test set!")

    multihead_model_path = os.path.join(
        MODEL_SAVE_PATH,
        f"vit_base_patch16_224_lr-3e-05_wd-0.001_lt-{lt}_bs-32_hd-256_d-0.3_a-silu_seed-8359_MultiHead.pth"
    )

    metrics = evaluate_multihead_models(
        multihead_model_path,
        device,
        lt
    )

    multihead_lambda_test_results.append(metrics)

In [ ]:
# Build summary DataFrames for testing results

single_model_summary_rows = []

single_model_summary_rows.append({
    "model": "Binary",
    "lambda_value": None,
    "real_ai_acc": binary_test_metrics["real_ai_acc"],
    "real_ai_precision": binary_test_metrics["real_ai_precision"],
    "real_ai_recall": binary_test_metrics["real_ai_recall"],
    "real_ai_f1": binary_test_metrics["real_ai_f1"],
    "transform_acc": None,
    "transform_macro_f1": None,
    "combined_score": None
})

single_model_summary_rows.append({
    "model": "Transformation",
    "lambda_value": None,
    "real_ai_acc": None,
    "real_ai_precision": None,
    "real_ai_recall": None,
    "real_ai_f1": None,
    "transform_acc": transformation_test_metrics["transform_acc"],
    "transform_macro_f1": transformation_test_metrics["transform_macro_f1"],
    "combined_score": None
})

multihead_summary_rows = []

for metrics in multihead_lambda_test_results:
    multihead_summary_rows.append({
        "model": "Multi-Head",
        "lambda_value": metrics["lambda_value"],
        "real_ai_acc": metrics["real_ai_acc"],
        "real_ai_precision": metrics["real_ai_precision"],
        "real_ai_recall": metrics["real_ai_recall"],
        "real_ai_f1": metrics["real_ai_f1"],
        "transform_acc": metrics["transform_acc"],
        "transform_macro_f1": metrics["transform_macro_f1"],
        "combined_score": metrics["combined_score"]
    })

df_single_model_test_summary = pd.DataFrame(single_model_summary_rows)
df_multihead_lambda_test_summary = pd.DataFrame(multihead_summary_rows)
df_all_test_summary = pd.concat(
    [
        df_single_model_test_summary,
        df_multihead_lambda_test_summary
    ],
    ignore_index=True
)

display(df_all_test_summary)

In [ ]:
# Build split metric DataFrames

real_ai_transform_rows = []
real_ai_class_transform_rows = []
transformation_split_rows = []

# Binary model Real/AI split results
for transform_name, values in binary_test_metrics["real_ai_accuracy_by_transformation"].items():
    real_ai_transform_rows.append({
        "model": "Binary",
        "lambda_value": None,
        "transformation": transform_name,
        "samples": values["samples"],
        "accuracy": values["accuracy"]
    })

for class_name, transform_results in binary_test_metrics[
    "real_ai_accuracy_by_class_and_transformation"
].items():
    for transform_name, values in transform_results.items():
        real_ai_class_transform_rows.append({
            "model": "Binary",
            "lambda_value": None,
            "class": class_name,
            "transformation": transform_name,
            "group": f"{class_name} | {transform_name}",
            "samples": values["samples"],
            "accuracy": values["accuracy"]
        })

# Transformation single-head split results
for transform_name, values in transformation_test_metrics[
    "transformation_accuracy_by_true_transformation"
].items():
    transformation_split_rows.append({
        "model": "Transformation",
        "lambda_value": None,
        "true_transformation": transform_name,
        "samples": values["samples"],
        "accuracy": values["accuracy"]
    })

# Multi-Head split results for every lambda
for metrics in multihead_lambda_test_results:
    lt = metrics["lambda_value"]

    for transform_name, values in metrics["real_ai_accuracy_by_transformation"].items():
        real_ai_transform_rows.append({
            "model": "Multi-Head",
            "lambda_value": lt,
            "transformation": transform_name,
            "samples": values["samples"],
            "accuracy": values["accuracy"]
        })

    for class_name, transform_results in metrics[
        "real_ai_accuracy_by_class_and_transformation"
    ].items():
        for transform_name, values in transform_results.items():
            real_ai_class_transform_rows.append({
                "model": "Multi-Head",
                "lambda_value": lt,
                "class": class_name,
                "transformation": transform_name,
                "group": f"{class_name} | {transform_name}",
                "samples": values["samples"],
                "accuracy": values["accuracy"]
            })

    for transform_name, values in metrics[
        "transformation_accuracy_by_true_transformation"
    ].items():
        transformation_split_rows.append({
            "model": "Multi-Head",
            "lambda_value": lt,
            "true_transformation": transform_name,
            "samples": values["samples"],
            "accuracy": values["accuracy"]
        })


df_real_ai_by_transformation = pd.DataFrame(real_ai_transform_rows)
df_real_ai_by_class_transformation = pd.DataFrame(real_ai_class_transform_rows)
df_transformation_by_true_transformation = pd.DataFrame(transformation_split_rows)

display(df_real_ai_by_transformation)
display(df_real_ai_by_class_transformation)
display(df_transformation_by_true_transformation)

In [ ]:
df_all_test_summary
df_real_ai_by_transformation
df_real_ai_by_class_transformation
df_transformation_by_true_transformation
binary_test_metrics
transformation_test_metrics
multihead_lambda_test_results

#Visualization


In [ ]:
# Visualization setup for testing results

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIGURE_SAVE_PATH = os.path.join(os.getcwd(), "presentation_figures")
os.makedirs(FIGURE_SAVE_PATH, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})


def save_figure(file_name):
    output_path = os.path.join(
        FIGURE_SAVE_PATH,
        f"{file_name}.png"
    )

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
    plt.show()


def add_bar_labels(ax, decimals=1):
    for container in ax.containers:
        labels = []

        for bar in container:
            height = bar.get_height()

            if np.isnan(height):
                labels.append("")
            else:
                labels.append(f"{height:.{decimals}f}")

        ax.bar_label(
            container,
            labels=labels,
            fontsize=8,
            padding=2
        )


def make_model_label(row):
    if row["model"] == "Multi-Head":
        return f"Multi-Head λ={row['lambda_value']}"
    return row["model"]

In [ ]:
# Visualization 1: Overall test metrics comparison

df_overall_plot = df_all_test_summary.copy()
df_overall_plot["model_label"] = df_overall_plot.apply(
    make_model_label,
    axis=1
)

metric_columns = [
    "real_ai_acc",
    "real_ai_f1",
    "transform_acc",
    "transform_macro_f1",
    "combined_score"
]

available_metric_columns = [
    column
    for column in metric_columns
    if column in df_overall_plot.columns and df_overall_plot[column].notna().any()
]

plot_df = (
    df_overall_plot
    .set_index("model_label")[available_metric_columns]
    * 100
)

display(plot_df)

ax = plot_df.plot(
    kind="bar",
    figsize=(13, 6)
)

ax.set_title("Final test performance comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.xticks(rotation=30, ha="right")
add_bar_labels(ax)

save_figure("test_overall_metric_comparison")

In [ ]:
# Visualization 2: Multi-Head performance across lambda values

df_lambda_plot = (
    df_all_test_summary[df_all_test_summary["model"] == "Multi-Head"]
    .copy()
    .sort_values("lambda_value")
)

display(df_lambda_plot)

plt.figure(figsize=(10, 6))

plt.plot(
    df_lambda_plot["lambda_value"],
    df_lambda_plot["real_ai_acc"] * 100,
    marker="o",
    linewidth=2,
    label="Real/AI accuracy"
)

plt.plot(
    df_lambda_plot["lambda_value"],
    df_lambda_plot["real_ai_f1"] * 100,
    marker="o",
    linewidth=2,
    label="Real/AI F1"
)

plt.plot(
    df_lambda_plot["lambda_value"],
    df_lambda_plot["transform_acc"] * 100,
    marker="o",
    linewidth=2,
    label="Transformation accuracy"
)

plt.plot(
    df_lambda_plot["lambda_value"],
    df_lambda_plot["combined_score"] * 100,
    marker="o",
    linewidth=2,
    label="Combined score"
)

best_lambda_row = df_lambda_plot.loc[
    df_lambda_plot["combined_score"].idxmax()
]

plt.title("Multi-Head test performance across lambda values")
plt.xlabel("Lambda transformation value")
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()

print(
    "Best lambda by combined score:",
    best_lambda_row["lambda_value"],
    "| Combined score:",
    f"{best_lambda_row['combined_score']:.4f}"
)

save_figure("multihead_lambda_metric_comparison")

In [ ]:
# Visualization 3: Real/AI accuracy by transformation

df_real_ai_transform_plot = df_real_ai_by_transformation.copy()

df_real_ai_transform_plot["model_label"] = df_real_ai_transform_plot.apply(
    lambda row: (
        "Binary"
        if row["model"] == "Binary"
        else f"Multi-Head λ={row['lambda_value']}"
    ),
    axis=1
)

pivot_real_ai_transform = (
    df_real_ai_transform_plot
    .pivot(
        index="model_label",
        columns="transformation",
        values="accuracy"
    )
    .reindex(columns=list(TRANSFORMATION_NAMES.values()))
    * 100
)

display(pivot_real_ai_transform)

ax = pivot_real_ai_transform.plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title("Real/AI test accuracy by transformation")
ax.set_xlabel("Model")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Transformation")

plt.xticks(rotation=30, ha="right")
add_bar_labels(ax)

save_figure("real_ai_accuracy_by_transformation")

In [ ]:
# Visualization 4: Multi-Head Real/AI accuracy by class, transformation, and lambda

df_mh_class_transform = (
    df_real_ai_by_class_transformation[
        df_real_ai_by_class_transformation["model"] == "Multi-Head"
    ]
    .copy()
)

group_order = [
    f"{class_name} | {transform_name}"
    for class_name in REAL_AI_NAMES.values()
    for transform_name in TRANSFORMATION_NAMES.values()
]

pivot_class_transform = (
    df_mh_class_transform
    .pivot(
        index="group",
        columns="lambda_value",
        values="accuracy"
    )
    .reindex(group_order)
    .sort_index(axis=1)
    * 100
)

display(pivot_class_transform)

matrix = pivot_class_transform.to_numpy()

fig, ax = plt.subplots(figsize=(10, 6))

image = ax.imshow(
    matrix,
    aspect="auto"
)

ax.set_title("Multi-Head Real/AI accuracy by class, transformation, and lambda")
ax.set_xlabel("Lambda transformation value")
ax.set_ylabel("Class and transformation")

ax.set_xticks(np.arange(len(pivot_class_transform.columns)))
ax.set_xticklabels(pivot_class_transform.columns)

ax.set_yticks(np.arange(len(pivot_class_transform.index)))
ax.set_yticklabels(pivot_class_transform.index)

for row_idx in range(matrix.shape[0]):
    for col_idx in range(matrix.shape[1]):
        value = matrix[row_idx, col_idx]

        if not np.isnan(value):
            ax.text(
                col_idx,
                row_idx,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontweight="bold"
            )

fig.colorbar(image, ax=ax, label="Accuracy (%)")

save_figure("multihead_real_ai_class_transformation_lambda_heatmap")

In [ ]:
# Visualization 5: Transformation accuracy by true transformation

df_transform_plot = df_transformation_by_true_transformation.copy()

df_transform_plot["model_label"] = df_transform_plot.apply(
    lambda row: (
        "Transformation"
        if row["model"] == "Transformation"
        else f"Multi-Head λ={row['lambda_value']}"
    ),
    axis=1
)

pivot_transform = (
    df_transform_plot
    .pivot(
        index="model_label",
        columns="true_transformation",
        values="accuracy"
    )
    .reindex(columns=list(TRANSFORMATION_NAMES.values()))
    * 100
)

display(pivot_transform)

ax = pivot_transform.plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title("Transformation-head accuracy by true transformation")
ax.set_xlabel("Model")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="True transformation")

plt.xticks(rotation=30, ha="right")
add_bar_labels(ax)

save_figure("transformation_accuracy_by_true_transformation")

In [ ]:
# Visualization 6: Confusion matrices for Binary, Transformation, and best Multi-Head model

def plot_confusion_matrix(cm, labels, title, file_name):
    matrix = np.asarray(cm)

    fig, ax = plt.subplots(figsize=(6, 5))

    image = ax.imshow(
        matrix,
        aspect="equal"
    )

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)

    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)

    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            ax.text(
                col_idx,
                row_idx,
                int(matrix[row_idx, col_idx]),
                ha="center",
                va="center",
                fontweight="bold"
            )

    fig.colorbar(image, ax=ax, label="Number of images")

    save_figure(file_name)


best_multihead_metrics = max(
    multihead_lambda_test_results,
    key=lambda metrics: metrics["combined_score"]
)

best_lambda_value = best_multihead_metrics["lambda_value"]

plot_confusion_matrix(
    binary_test_metrics["real_ai_confusion_matrix"],
    ["Real", "AI"],
    "Binary model Real/AI confusion matrix",
    "binary_real_ai_confusion_matrix"
)

plot_confusion_matrix(
    transformation_test_metrics["transform_confusion_matrix"],
    [name.title() for name in TRANSFORMATION_NAMES.values()],
    "Single-head transformation confusion matrix",
    "singlehead_transformation_confusion_matrix"
)

plot_confusion_matrix(
    best_multihead_metrics["real_ai_confusion_matrix"],
    ["Real", "AI"],
    f"Best Multi-Head Real/AI confusion matrix, λ={best_lambda_value}",
    "best_multihead_real_ai_confusion_matrix"
)

plot_confusion_matrix(
    best_multihead_metrics["transform_confusion_matrix"],
    [name.title() for name in TRANSFORMATION_NAMES.values()],
    f"Best Multi-Head transformation confusion matrix, λ={best_lambda_value}",
    "best_multihead_transformation_confusion_matrix"
)

In [ ]:
# Visualization 7: Best Multi-Head lambda summary

best_multihead_summary = pd.DataFrame([
    {
        "best_lambda_value": best_multihead_metrics["lambda_value"],
        "real_ai_acc": best_multihead_metrics["real_ai_acc"],
        "real_ai_precision": best_multihead_metrics["real_ai_precision"],
        "real_ai_recall": best_multihead_metrics["real_ai_recall"],
        "real_ai_f1": best_multihead_metrics["real_ai_f1"],
        "transform_acc": best_multihead_metrics["transform_acc"],
        "transform_macro_f1": best_multihead_metrics["transform_macro_f1"],
        "combined_score": best_multihead_metrics["combined_score"]
    }
])

display(best_multihead_summary)

best_metrics_plot = (
    best_multihead_summary
    .drop(columns=["best_lambda_value"])
    .iloc[0]
    * 100
)

ax = best_metrics_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    f"Best Multi-Head model test metrics, λ={best_multihead_metrics['lambda_value']}"
)
ax.set_xlabel("Metric")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)

for bar in ax.patches:
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 1,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xticks(rotation=30, ha="right")

save_figure("best_multihead_lambda_summary")